# Day 9

In [1]:
import fs from "node:fs";

In [2]:
const input = fs.readFileSync("input.txt", "utf-8");

In [3]:
const sample = `\
7,1
11,1
11,7
9,7
9,5
2,5
2,3
7,3`

## Part 1
Largest rectangle spanned by diagonal.

In [6]:
const process = input => input.split('\n').map(r => r.split(',').map(n => parseInt(n)));

In [8]:
const coords = process(sample);
coords;

[
  [ 7, 1 ],  [ 11, 1 ],
  [ 11, 7 ], [ 9, 7 ],
  [ 9, 5 ],  [ 2, 5 ],
  [ 2, 3 ],  [ 7, 3 ]
]

Now, the area between two points should satsify:

- `A([2,5], [9,7]) = 24`

`A([x0, y0], [x, y1]) = (x1 - x0 + 1)*(y - y0 + 1)`

In [9]:
const A = (p0, p1) => (Math.abs(p0[0] - p1[0]) + 1) * (Math.abs(p0[1] - p1[1]) + 1);
A(coords[3], coords[5])

24

In [11]:
let largest = 0;
for (let i = 0; i < coords.length; i++) {
  const p0 = coords[i];
  for (let j = i+1; j < coords.length; j++) {
    const p1 = coords[j];
    largest = Math.max(largest, A(p0, p1));
  }
}
largest

50

In [14]:
function part1(input) {
  const coords = process(input);
  let largest = 0;
  for (let i = 0; i < coords.length; i++) {
    const p0 = coords[i];
    for (let j = i+1; j < coords.length; j++) {
      const p1 = coords[j];
      largest = Math.max(largest, A(p0, p1));
    }
  }
  return largest;
}
part1(sample);

50

In [15]:
part1(input);

4746238001

## Part 2
Now we add a constraint on the rectangles. The input defines a polygon, and the rectangle must be contained within the polygon. This seems trickier, and I'm also not too familiar with computational geometry algorithms. Therefore we have more of an exploration today, and perhaps something we'll come of it.

Let's start asking ourselves some questions. For example, how can we tell when a point is inside a rectangle?, let's start by creating some visualisation tools

In [19]:
const board_size = 9;
const board = [];
for (let i = 0; i < board_size; i++) {
  board.push([]);
  for (let j = 0; j < board_size; j++) {
    board[i].push('.');
  }
}
console.log(board.map(r => r.join('')).join('\n'))

.........
.........
.........
.........
.........
.........
.........
.........
.........


In [51]:
function empty_board(h, w) {
  const board = [];
  for (let i = 0; i < h; i++) {
    board.push([]);
    for (let j = 0; j < w; j++) {
      board[i].push('.');
    }
  }
  return board;
}

function render_board(board) {
  console.log(board.map(r => r.join('')).join('\n'));
}

function visualize(pts, rects, board_size) {
  const board = empty_board(...board_size);
  for (const r of rects) {
    const [x0, x1] = [r[0][0], r[1][0]].sort((a, b) => a - b);
    const [y0, y1] = [r[0][1], r[1][1]].sort((a, b) => a - b);
    for (let x = x0; x <= x1; x++) {
      for (let y = y0; y <= y1; y++) {
        board[y][x] = "O";
      }
    }
  }
  for (const [x,y] of pts) {
    board[y][x] = "#";
  }
  render_board(board);
}
visualize(coords, [[coords[3],coords[5]]], [9,14]);

..............
.......#...#..
..............
..#....#......
..............
..#OOOOOO#....
..OOOOOOOO....
..OOOOOOO#.#..
..............


Okay, got some vis. So when is a point `x,y` is inside a rectangle `x0, y0, x1, y1`? of course when

- `x0 <= x <= x1`
- `y0 <= y <= y1`

A harder problem is to check when a segment crosses into a rectangle. Now the problem let's us assume that the sides of the polygons are all aligned to one of axes. This means we need to consider horizontal or vertical segments.

Hmm, I have another idea. First we need to answer a similar question: Given the polygon boundary and a point, how do we determine whether the point is in or out of the polygon? This reminds me of integrating a curve in a vector field: if the curve contains the "source" of the vector field then the integral should be non-negative. 

Consulting with claude a bit, it suggests ray casting, shooting a horizontal ray and counting how many times it crosses the polygon edges. I think I was using it in 2023 but forgot about it. Anyhow, let's try to implement it. First, how do we know when to segments intersect? since we'll use a horizontal ray, we can consider vertical rays. We then need to think about the special case where the point is on the edge.

A horizontal ray to the right from point `(x,y)` intersects a vertical segment `(xs,y0, y1)` if `x<=xs` and `y0<=y<=y1`

In [ ]:
let inside = false;